In [56]:
import pandas as pd
import numpy as np
import re
import math

In [57]:
pepDB = pd.read_csv('peptide_data/peptide_db.csv')
polySampDB = pd.read_csv('polymer_data/polymer_samples_db.csv')

In [58]:
pepDB['ID'] = pepDB.apply(lambda row: 'pepID' + str(row['pep_ID']), axis=1)
polySampDB['ID'] = polySampDB.apply(lambda row: 'polyID' + str(row['poly_ID']) + '_S' + str(row['sample_ID']), axis=1)

## Combine Polymer and Peptide DBs

In [59]:
df = pd.concat([pepDB, polySampDB], ignore_index=True).dropna(axis=1)
df.loc[df['MIC_ecoli'] > 1000, 'MIC_ecoli'] = 1000
df

,sequence,MIC_ecoli,ID
0,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,125.878150,pepID1
1,AAAAAAAIKMLMDLVNERIMALNKKAKK_amd,10.000000,pepID2
2,AAAAGSVWGAVNYTSDCNGECKRRGYKGGYCGSFANVNCWCET,100.000000,pepID3
3,AAAKAALNAVLVGANA,80.000000,pepID4
4,AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,1.780176,pepID5
...,...,...,...
16198,BmamTmaTmaBmamTmaTmaTmaBmamOlamTmaTmaOlamOlamT...,2.471042,polyID52_S96
16199,TmaTmaTmaTmaTmaTmaTmaTmaTmaBmamTmaTmaBmamPheTm...,2.471042,polyID52_S97
16200,TmaTmaTmaTmaTmaPheTmaTmaBmamMepBmamTmaOlamTmaT...,2.471042,polyID52_S98
16201,BmamMepPheTmaTmaTmaTmaTmaTmaTmaTmaTmaTmaOlamTm...,2.471042,polyID52_S99


## Assign Peptide or Polymer to Class Based on MIC Value

In [60]:
def bin(num_classes):

    base = np.exp(np.log(1024.1)/num_classes)
    bins = [0]

    for i in range(num_classes):
        bins.append(base**(i+1))

    return bins

In [61]:
for i in range(2,11):
    bin_vals = bin(i)
    labels = range(0, len(bin_vals)-1)  # Labels 1 through 11
    # reverse order: small--> max bin, large --> min_bin
    df[str(i) + '_classes'] = pd.cut(df['MIC_ecoli'], bins=bin_vals, labels=range(i-1, -1, -1), right=True)

# find all columns whose name contains 'class'
class_cols = df.columns[df.columns.str.contains('class')]

# # convert them in place to int
df[class_cols] = df[class_cols].astype(int)
# df[df.isna().any(axis=1)]

In [62]:
df.to_csv('db.csv', index=False)